In [1]:
from pathlib import Path
import inspect
import json
import random
import sys

import numpy as np
import pandas as pd

def resolve_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'fbn_pipeline').exists():
            return candidate
    raise RuntimeError('Project root not found')

PROJECT_ROOT = resolve_project_root()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from fbn_pipeline import (
    ExperimentConfig,
    apply_preset,
    lock_development_config,
    run_development,
    run_final_evaluation_from_lock,
    select_best_development_config,
)
from fbn_pipeline.data import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('PROTOCOL MODE = DEVELOPMENT_ONLY')
print(f'PROJECT_ROOT = {PROJECT_ROOT}')


PROTOCOL MODE = DEVELOPMENT_ONLY
PROJECT_ROOT = C:\Users\doqua\Desktop\fuzzy-bayesian-cognitive-load\fts-anfis-bns-final


In [2]:
dataset = load_dataset(PROJECT_ROOT / 'last_30s_segments')
X_train = dataset.X_train
y_train = dataset.y_train
user_train = dataset.user_train
print('Loaded training split only for smoke validation')
print(f'X_train shape = {X_train.shape}')
print(f'y_train shape = {y_train.shape}')
print(f'unique training subjects = {sorted(np.unique(user_train).tolist())}')


Loaded training split only for smoke validation
X_train shape = (632, 4, 30)
y_train shape = (632,)
unique training subjects = ['1mpau', '2nxs5', '5gpsc', '7swyk', '8a1ep', 'b7mrd', 'c24ur', 'dkhty', 'e4gay', 'ef5rq', 'f3j25', 'hpbxa', 'ibvx8', 'iz2ps', 'rc1in', 'tn4vl', 'wjxci', 'yljm5']


In [3]:
print('SMOKE VALIDATION ONLY - NOT PUBLICATION SELECTION')
candidate_specs = [
    ('EXP2_RULES_3', apply_preset(ExperimentConfig(data_path='last_30s_segments'), 'EXP2_RULES_3')),
    ('EXP2_RULES_6', apply_preset(ExperimentConfig(data_path='last_30s_segments'), 'EXP2_RULES_6')),
]
print([(candidate_id, cfg.n_rules, cfg.n_sets, cfg.n_bins) for candidate_id, cfg in candidate_specs])


SMOKE VALIDATION ONLY - NOT PUBLICATION SELECTION
[('EXP2_RULES_3', 3, 3, 5), ('EXP2_RULES_6', 6, 3, 5)]


In [4]:
development_results = []
for candidate_id, cfg in candidate_specs:
    result = run_development(
        X_train,
        y_train,
        user_train,
        cfg,
        configuration_id=candidate_id,
        print_progress=True,
    )
    development_results.append(result)

development_table = pd.DataFrame([
    {
        'config_id': result.configuration_id,
        'oof_accuracy': result.oof_metrics['accuracy'],
        'oof_balanced_accuracy': result.oof_metrics['bal_acc'],
        'oof_macro_f1': result.oof_metrics['macro_f1'],
        'oof_f1': result.oof_metrics['f1'],
        'oof_auc': result.oof_metrics['auc'],
        'oof_threshold': result.threshold,
    }
    for result in development_results
])
print(development_table.to_string(index=False))



[development] Running GroupKFold CV on training subjects only...
--------------------------------------------------------------------------------
  Fold | Split | Total | cls 0 | cls 1 | Users
--------------------------------------------------------------------------------
     1 | Train |   504 |   251 |   253 | ['1mpau', '2nxs5', '7swyk', '8a1ep', 'b7mrd', 'c24ur', 'dkhty', 'e4gay', 'ef5rq', 'f3j25', 'rc1in', 'tn4vl', 'wjxci', 'yljm5']
       |   Val |   128 |    66 |    62 | ['5gpsc', 'hpbxa', 'ibvx8', 'iz2ps']
--------------------------------------------------------------------------------
  Fold  1 | Train | Acc: 0.6647 | AUC: 0.7229 | F1: 0.7140
         |  Val  | Acc: 0.4922 | AUC: 0.5174 | F1: 0.5578
--------------------------------------------------------------------------------
     2 | Train |   517 |   260 |   257 | ['2nxs5', '5gpsc', '7swyk', '8a1ep', 'b7mrd', 'c24ur', 'dkhty', 'e4gay', 'ef5rq', 'f3j25', 'hpbxa', 'ibvx8', 'iz2ps', 'rc1in', 'tn4vl']
       |   Val |   115 

In [5]:
selection = select_best_development_config(development_results)
selected = selection.selected_result
selection_summary = {
    'primary_metric': selection.ranking_metric,
    'tie_break_rules': selection.tie_break_rules,
    'selected_smoke_candidate': selected.configuration_id,
    'selected_threshold': selected.threshold,
    'selected_oof_metrics': selected.oof_metrics,
}
print(json.dumps(selection_summary, indent=2))


{
  "primary_metric": "oof_balanced_accuracy",
  "tie_break_rules": [
    "oof_macro_f1",
    "oof_auc",
    "lower_model_complexity",
    "configuration_id_lexicographic"
  ],
  "selected_smoke_candidate": "EXP2_RULES_6",
  "selected_threshold": 0.55,
  "selected_oof_metrics": {
    "accuracy": 0.6234,
    "precision": 0.6091,
    "recall": 0.6825,
    "f1": 0.6437,
    "auc": 0.6201,
    "acc": 0.6234,
    "bal_acc": 0.6236,
    "macro_f1": 0.6222
  }
}


In [6]:
lock_path = PROJECT_ROOT / 'artifacts' / 'protocol_v2_smoke' / 'smoke_locked_config.json'
lock_payload = lock_development_config(
    selection.selected_result,
    selection,
    output_path=lock_path,
)
loaded_lock = json.loads(lock_path.read_text(encoding='utf-8'))
required_fields = {
    'protocol_version', 'configuration_id', 'selected_using', 'threshold_source',
    'threshold', 'config', 'oof_metrics', 'final_test_evaluated', 'config_hash', 'lock_hash'
}
missing_fields = sorted(required_fields.difference(loaded_lock))
assert not missing_fields, missing_fields
assert loaded_lock['selected_using'] == 'training_oof_only'
assert loaded_lock['threshold_source'] == 'training_oof_only'
assert loaded_lock['final_test_evaluated'] is False
assert loaded_lock['config_hash'] == lock_payload['config_hash']
assert loaded_lock['lock_hash'] == lock_payload['lock_hash']
print(json.dumps({'lock_path': str(lock_path), 'validated_fields': sorted(required_fields)}, indent=2))


{
  "lock_path": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\protocol_v2_smoke\\smoke_locked_config.json",
  "validated_fields": [
    "config",
    "config_hash",
    "configuration_id",
    "final_test_evaluated",
    "lock_hash",
    "oof_metrics",
    "protocol_version",
    "selected_using",
    "threshold",
    "threshold_source"
  ]
}


In [7]:
signature = inspect.signature(run_final_evaluation_from_lock)
signature_text = str(signature)
assert 'locked_config_path' in signature.parameters
assert 'dataset' in signature.parameters
assert 'config' not in signature.parameters
assert 'kwargs' not in signature.parameters

missing_lock_error = None
try:
    run_final_evaluation_from_lock(
        locked_config_path=PROJECT_ROOT / 'artifacts' / 'protocol_v2_smoke' / 'missing.json',
        dataset=None,
        confirm_final_evaluation=True,
    )
except Exception as exc:
    missing_lock_error = f'{type(exc).__name__}: {exc}'

malformed_lock_path = PROJECT_ROOT / 'artifacts' / 'protocol_v2_smoke' / 'malformed_lock.json'
malformed_lock_path.write_text(json.dumps({'protocol_version': 'publication_protocol_v2'}), encoding='utf-8')
malformed_lock_error = None
try:
    run_final_evaluation_from_lock(
        locked_config_path=malformed_lock_path,
        dataset=None,
        confirm_final_evaluation=True,
    )
except Exception as exc:
    malformed_lock_error = f'{type(exc).__name__}: {exc}'

assert missing_lock_error is not None
assert malformed_lock_error is not None
print(json.dumps({
    'signature': signature_text,
    'missing_lock_error': missing_lock_error,
    'malformed_lock_error': malformed_lock_error,
}, indent=2))


{
  "signature": "(*, locked_config_path: 'Path | str', dataset: 'DatasetSplit', output_dir: 'Path | str | None' = None, confirm_final_evaluation: 'bool' = False, final_csv_path: 'Path | str | None' = None) -> 'FinalEvaluationResult'",
  "missing_lock_error": "FileNotFoundError: Locked configuration not found: C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\protocol_v2_smoke\\missing.json",
  "malformed_lock_error": "ValueError: Locked configuration is missing required fields: ['config', 'config_hash', 'configuration_id', 'final_test_evaluated', 'grouping_protocol', 'k_folds', 'lock_hash', 'oof_metrics', 'schema_version', 'seed', 'selected_using', 'selection_metric', 'threshold', 'threshold_source', 'tie_break_rules']"
}


In [8]:
summary = {
    'development_only_run_succeeded': True,
    'oof_metrics_generated': all(result.oof_metrics for result in development_results),
    'ranking_used_oof_only': selection.ranking_metric == 'oof_balanced_accuracy',
    'temporary_lock_created': lock_path.exists(),
    'final_test_evaluation_executed': False,
    'test_metrics_generated': False,
    'final_evaluation_api_exists_but_unexecuted': callable(run_final_evaluation_from_lock),
}
print(json.dumps(summary, indent=2))


{
  "development_only_run_succeeded": true,
  "oof_metrics_generated": true,
  "ranking_used_oof_only": true,
  "temporary_lock_created": true,
  "final_test_evaluation_executed": false,
  "test_metrics_generated": false,
  "final_evaluation_api_exists_but_unexecuted": true
}
